### Generate Scene Graphs:

In [1]:
%load_ext autoreload
%autoreload 2
import os
import sys
import json
sys.path.append('/Volumes/scratch/alegretelena/LLaVA-3D/open3dsg') # sys.path.append('../../LLaVA-3D/open3dsg')
import re
import torch
import numpy as np
from tqdm import tqdm
from graphviz import Digraph
import matplotlib.colors as mcolors
from open_dataset import Open2D3DSGDataset

import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

CONF_PATH_R3SCAN_RAW = '/Volumes/projects/open3dsg/data/3RScan/data/' # '/home/lodes/uni/3.semester/project/data/3rscan/raw'
CONF_PATH_R3SCAN_PROCESSED = '../../LLaVA-3D/data/3rscan/processed/' # '/home/lodes/uni/3.semester/project/data/3rscan/processed'
obj_class_dict = [line.rstrip() for line in open(os.path.join("/Volumes/projects/open3dsg/data/3RScan/", "3DSSG_subset", "classes.txt"), "r").readlines()]

# Define colors
# colors = list(mcolors.TABLEAU_COLORS.keys())
# node_color_list = list(mcolors.TABLEAU_COLORS.values())

/Users/elenaalegretregalado/miniconda3/envs/lap_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Define colors: 
- colors: ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown', 'tab:pink', 'tab:gray', 'tab:olive', 'tab:cyan']

- node_color_list: ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']


Relationships:
- relationships_{mode}.json: ['scan', 'objects', 'relationships', 'split']

Objects: 
- objects_json: ['scan', 'objects']
- objects_json objects: ['ply_color', 'nyu40', 'eigen13', 'label', 'rio27', 'affordances', 'id', 'global_id', 'attributes']


### Helper Methods

In [2]:
def vis_graph_scannet_clip(scan_id, model, objects_gt, gt_predicates, predicates, edges, objects_color, filename='graph', include_nones = False):
    # Create a directed graph using Graphviz
    dot = Digraph(comment='The Scene Graph')
    dot.attr(rankdir='TB') # Set the graph layout to flow from top to bottom

    # Graph label == scan ID
    dot.attr(label=scan_id)
    dot.attr('node', shape='oval', fontname='Sans')  # Default node attributes (oval shape, Sans font)

    # Split the scan ID into two parts
    a = scan_id
    a, b = '-'.join(a.split('-')[:-1]), a.split('-')[-1]
    # g_colors = node_color_list  # {o['id']: o['ply_color'] for o in self.scene_graphs_val[a+'_'+b]['objects']}
    
    # Loop through the list of ground truth objects to add nodes to the graph
    for index in range(len(objects_gt)):
        id = str(index)# Use the index as the node ID
        dot.attr('node', fillcolor=objects_color[index], style='filled')  # Set node color and fill style

        #pred = obj_class_dict[objects_gt[index]]
        # Get the object name
        pred = objects_gt[index]
        pred = pred if pred != 'socket' else 'wall'
        #dot.node(id, pred+f" [{objects_gt[index]}] "+'-'+str(object_ids[index].item()))
        # Add a node to the graph with the specified ID and label (object name)
        dot.node(id, pred)

    # Set edge attributes (black color, filled style, Sans font)
    dot.attr('edge', fontname='Sans', color='black', style='filled')
    # Loop through the ground truth predicates and edges to add relationships to the graph
    for i, (gt_predicate, edge) in enumerate(zip(gt_predicates, edges)):
        # Skip processing if the ground truth predicate is 'none'
        if gt_predicate == "none":
            break

        # Extract source and destination nodes from the edge
        # Get the predicted relationship from the model
        s, o = edge[:2]
        p_s = predicates[i]

        # Check if the predicted relationship is irrelevant and skip if include_nones is False
        if np.array([none_p in p_s for none_p in ['and',  'unrelated', 'not', 'none']]).any() and not include_nones:
            if p_s in ['', 'and', ' ', 'unrelated', 'not', 'none']:
                continue
        # Create a label showing both the ground truth and model-predicted relationships
        label = "GT: " + gt_predicate + "\n" + model + ": " + p_s
        
        # Add an edge between the source and destination nodes with the label
        dot.edge(str(s.item()), str(o.item()), label)
        
    # Render the graph to a PNG file and clean up temporary files
    dot.render(filename, format="png", cleanup=True)

In [3]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

def visualize_relationship_embeddings(rel_gt_encoded, rel_gt_labels, pred_encoded, pred_label, gt_labels, rel_mapper, method='pca'):
    """
    Visualizes relationship embeddings in a 2D space.

    Args:
        gt_embeddings (torch.Tensor): All ground truth relationship embeddings (shape: [N, 768]).
        gt_labels (list of str): Labels for all ground truth embeddings.
        rel_gt_encoded (torch.Tensor): Ground truth embeddings for the current prediction (shape: [M, 768]).
        rel_gt_labels (list of str): Labels for the ground truth embeddings related to the current prediction.
        pred_encoded (torch.Tensor): Predicted relationship embedding (shape: [768]).
        pred_label (str): Label for the predicted relationship.
        method (str): Dimensionality reduction method ('pca' or 'tsne').
    """
    pastel_colors = {
        "gt": "#012E40",         # Blue
        "rel_gt": "#03A64A",     # Light blue
        "pred": "#F28705"        # Orange
        }

    gt_embeddings = F.normalize(torch.from_numpy(rel_mapper.encode(gt_labels)), dim=-1)

    # Normalize all embeddings
    gt_embeddings = F.normalize(gt_embeddings, dim=1)
    rel_gt_encoded = F.normalize(rel_gt_encoded, dim=1)
    pred_encoded = F.normalize(pred_encoded, dim=0)

    # Combine embeddings for dimensionality reduction
    all_embeddings = torch.cat((gt_embeddings, rel_gt_encoded, pred_encoded.unsqueeze(0)), dim=0)

    # Apply dimensionality reduction
    if method == 'pca':
        reducer = PCA(n_components=2)
    elif method == 'tsne':
        reducer = TSNE(n_components=2, random_state=42)
    else:
        raise ValueError("Invalid method. Use 'pca' or 'tsne'.")

    reduced_embeddings = reducer.fit_transform(all_embeddings.numpy())

    # Separate the embeddings into their respective groups
    num_gt = gt_embeddings.size(0)
    num_rel_gt = rel_gt_encoded.size(0)

    gt_points = reduced_embeddings[:num_gt]                         # Ground truth points (all relationships)
    rel_gt_points = reduced_embeddings[num_gt:num_gt + num_rel_gt]  # Ground truth points for the current prediction
    pred_point = reduced_embeddings[-1]                             # Prediction point

    # Create the plot
    plt.figure(figsize=(10, 8))

    # Plot all ground truth points (all relationships)
    plt.scatter(gt_points[:, 0], gt_points[:, 1], color=pastel_colors['gt'], label='All Ground Truth Relationships', alpha=0.6)

    # Add labels for all ground truth points
    for i, label in enumerate(gt_labels):
        plt.text(gt_points[i, 0], gt_points[i, 1], label, fontsize=10, ha='right', color=pastel_colors['gt'])

    # Plot ground truth points for the current prediction
    plt.scatter(rel_gt_points[:, 0], rel_gt_points[:, 1], color=pastel_colors['rel_gt'], label='Relevant Ground Truth Relationships', alpha=0.8)

    # Add labels for relevant ground truth points
    for i, label in enumerate(rel_gt_labels):
        plt.text(rel_gt_points[i, 0], rel_gt_points[i, 1], label, fontsize=10, ha='right', color=pastel_colors['rel_gt'])

    # Plot the prediction point
    plt.scatter(pred_point[0], pred_point[1], color=pastel_colors['pred'], label='Prediction', alpha=0.9)
    plt.text(pred_point[0], pred_point[1], pred_label, fontsize=12, ha='left', color=pastel_colors['pred'])

    # Configure the plot
    plt.title("Relationship Embeddings Visualization")
    plt.xlabel("Component 1")
    plt.ylabel("Component 2")
    plt.legend(loc='best')
    plt.grid(True)
    plt.show()


### Define the Dataset 

In [4]:
if __name__ == '__main__':
    print('Setting up the variables...')
    mode = 'train'
    relationship_list = []
    relationships = json.load(open(f"/Volumes/scratch/alegretelena/LLaVA-3D/data/relationships_{mode}.json"))["scans"] 

    print('Reading train_scans.txt...')
    with open(f'/Volumes/scratch/alegretelena/LLaVA-3D/data/{mode}_scans.txt', 'r') as file:
        scans_train = [line.strip() for line in file]
    for r in relationships:
        if r['scan'] in scans_train and r['split'] == 1:
            relationship_list.append(r)

    print('Defining the dataset...')       
    dataset = Open2D3DSGDataset(
        relationships_R3SCAN=relationship_list,
        relationships_scannet=None,
        path_3rscan_raw = CONF_PATH_R3SCAN_RAW,
        path_3rscan_processed = CONF_PATH_R3SCAN_PROCESSED,
        openseg=False,
        img_dim=224,
        rel_img_dim=224,
        top_k_frames=5,
        scales=3,
        mini=False,
        load_features=None,
        blip=True,
        llava=False,
        half=False,
        max_objects=9,
        max_rels=72)

    print('Done!')

Setting up the variables...
Reading train_scans.txt...
Defining the dataset...


100%|██████████| 15/15 [01:22<00:00,  5.53s/it]


Done!


### Scans to Index
Accessing data through indexed structures improves speed.

In [5]:
scan_id2idx = {}
for idx, data_sample in tqdm(enumerate(dataset)):
    scan_id2idx[data_sample['scan_id'][:-2]] = idx

scan_id2idx

15it [01:53,  7.57s/it]


{'0cf75f4b-564d-23f5-899f-ff97701d3f3b': 0,
 '4fbad32b-465b-2a5d-8499-85100e88f454': 1,
 'c92fb57e-f771-2064-86e4-5f4c7c77a8c7': 2,
 '752cc5a3-920c-26f5-8ff3-49518eff94c6': 3,
 '4acaebc0-6c10-2a2a-852e-0226d6539299': 4,
 'c12890e0-d3df-2d0d-87f7-9b8b04f663a5': 5,
 '198aaa76-0ba3-26f6-84de-6c13263a60bc': 6,
 'f3d7fa58-2835-2805-83bc-d2c583045bb4': 7,
 '5ed77dd4-c4f5-27a0-8476-f3048ea53ef5': 8,
 '422885dc-192d-25fc-857a-0c4af3695e4b': 9,
 '751a558c-fe61-2c3b-8f4e-340ddb43b8bd': 10,
 '77941462-cfdf-29cb-85a6-eb23498f9206': 11,
 '0cf75f50-564d-23f5-8a6b-cf1f98afcbce': 12,
 '8f0f1463-55de-28ce-80e5-d3294f7795ba': 13,
 '531cff08-0021-28f6-8e08-ba2eeb945e09': 14}

### Generate Graph using Cos Similarity or first element
When multiple relationships exist between objects in a scene, use cosine similarity to select the most similar one. The goal is to encode relationships as embeddings and find the most semantically similar one to the model's predicted relationship.

In [6]:
def process_relationships(data_dict, model, generated_texts, results_json, object_dict, eval_method='cos_similarity', visualitzation=False):
    # Load object class names from the classes.txt file
    obj_class_dict = [line.rstrip() for line in open(os.path.join(CONF_PATH_R3SCAN_RAW, "classes.txt"), "r").readlines()] # [..., 'armchair', ...]
    obj_count = data_dict['objects_count'].item()                                                                         # #num_obj
    rel_count = int(data_dict["predicate_count"].item())                                                                  # #num_relationships
    objects_gt = data_dict['objects_cat']                                                                                 # [118. 118. 139. 118. 104. 104.  57. 154.  33.]
    edges = data_dict['edges'][:rel_count]                                                                                #  edges: [..., [7 6], ...]
    object_edges = np.array(objects_gt[:obj_count][edges], dtype=np.int32)                                                # object_edges: [..., ['wall' 'floor'], ...]
    object_edges = np.array(obj_class_dict)[object_edges]
    object_edges[object_edges == 'socket'] = 'wall'

    predicates, objects, results_relationships = [], [], []
    if model == "blip":
        results = [generated_texts[i].rstrip().split(':')[-1].lstrip() for i in range(len(generated_texts))]
        results = [result if result != 'No relationship' else 'none' for result in results]
    elif model == "llava":
        qs = [result['query'] for result in results_json]
        results = [qs[i].split(':')[-1] + ' ' + generated_texts[i].rstrip().split('.')[0].split('\n')[0] for i in range(len(generated_texts))]
        results = [result if 'No relationship' not in result else 'none' for result in results]
        """
        Example: qs: Describe the relationship between the wall and the floor. Start the response with: the wall
                results:  the wall is above the floor
        """

    results = np.array(results)
    results = results.tolist()

    results_pred = [] # [..., ' is above ', ' is on ', ...]
    for i, (r, objs) in enumerate(zip(results, object_edges)):
        if not (objs[0] in r and objs[1] in r): # If str obj not in result --> None
            results_pred.append('none')
        else:
            try:
                results_pred.append(re.search(f'{objs[0]}(.*){objs[1]}', r).group(1).replace('the ', ''))
            except:
                results_pred.append('none')


    objects = object_edges                                              # [..., ['wall' 'floor'], ...]
    results_relationships.extend(results)                               # [..., ' the wall is above the floor', ...]
    predicates.extend(results_pred)                                     # [..., ' is on ', ...]

    data_dict['edges'] = torch.tensor(data_dict['edges']).unsqueeze(0)  #  edges: [..., [7 6], ...]
    predicate_count = data_dict['predicate_count'].item()               # predicate_count: #num_rel

    # Get the first relationship that appears in the GT for object i  
    objects_gt, objects_color = [], []
    for i in torch.unique(data_dict['edges']):                      # Get the relationship between edges without repeating it. 
        mask = (data_dict['edges'] == i).cpu()[0][:predicate_count] # Mask for the current edge. 
        objects_gt.append(objects[mask][0])                         # Get the str obj name that edge i (the objecti) has relationship with objj
        object_id = data_dict['objects_id'][i].astype('int')
        for obj in object_dict['objects']:                          # Get the color of the current obj
            if obj['id'] == str(object_id):
                objects_color.append(obj['ply_color'])
                break
        else:
            print('object_id: ', object_id)
            objects_color.append('#ffffff')
        
    pred_class_dict_orig = [line.rstrip() for line in open(os.path.join(CONF_PATH_R3SCAN_RAW, "relationships.txt"), "r").readlines()]
    
    if eval_method=='first_element':
        gt_predicates = [pred_class_dict_orig[predicates[0]] for predicates in data_dict['predicate_edges'] if predicates] # data_dict['predicate_edges']: [..., [rel1, ..., reli], ...]
    elif eval_method=='cos_similarity':
        # Get BERT encoder model
        rel_mapper = AutoModel.from_pretrained('jinaai/jina-embeddings-v2-base-en', trust_remote_code=True) #, cache_dir=cache_dir)

        gt_predicates = []

        # Iterate through relationships lst for a pair of objects.
        for idx, relationships in enumerate(data_dict['predicate_edges']):
            if relationships:
                # > 1 rel
                if len(relationships) > 1: 
                    # Encode GT rel
                    rel_words = [pred_class_dict_orig[i] for i in relationships]     # Convert rel_idx to human-readable rel
                    rel_gt_encoded = F.normalize(torch.from_numpy(rel_mapper.encode(rel_words)), dim=-1)

                    # Encode the predicted rel
                    pred_encoded = F.normalize(torch.from_numpy(rel_mapper.encode(predicates[idx])), dim=-1)
                    
                    # Calculate cosine similarity between predicted and ground truth embeddings
                    similarity_scores = torch.matmul(rel_gt_encoded, pred_encoded) # rel_gt_encoded @ pred_encoded.T
                    most_similar_idx = torch.argmax(similarity_scores).item()      # most similar rel idx 
                    most_similar_relationship = rel_words[most_similar_idx]        # Obtain the str name of the rel
                    gt_predicates.append(most_similar_relationship)
                    
                    if visualitzation: 
                        visualize_relationship_embeddings(rel_gt_encoded, rel_words, pred_encoded, predicates[idx], gt_labels=pred_class_dict_orig, rel_mapper=rel_mapper, method='pca')

                # == 1 rel
                else:
                    gt_predicates.append(pred_class_dict_orig[relationships[0]])
    return objects_gt, gt_predicates, predicates, objects_color

In [ ]:
# Load  objects.json file and get the list of scans
with open('/Volumes/projects/open3dsg/data/3RScan/3DSSG/objects.json') as file:
    objects_json = json.load(file)['scans']

model = 'llava'                              # Select model
result_type = 'llava_prompt'
for scan_id in scan_id2idx:
    data_dict = dataset[scan_id2idx.get(scan_id)]
    for obj in objects_json:
        if obj['scan'] == scan_id:
            object_dict = obj

    graph_filename = f"graphs/{scan_id}_{model}" # Define the graph image name

    # Load results.json file
    results_json = json.load(open(os.path.join(f"../results/{result_type}/results_{scan_id}_{model}.json")))
    generated_texts = [result['output'] for result in results_json]

    objects_gt, gt_predicates, predicates, objects_color = process_relationships(data_dict, model, generated_texts, results_json, object_dict, eval_method='cos_similarity', visualitzation=False)
    vis_graph_scannet_clip(data_dict['scan_id'], model, objects_gt, gt_predicates, predicates, data_dict['edges'][0][:len(predicates)], objects_color, filename=graph_filename, include_nones=True)

#### Print two graphs in the same graph

In [7]:
def vis_graph_two_graphs_clip(scan_id, objects_gt, gt_predicates, objects_color, edges, model1, predicates1, model2, predicates2, filename='graph', include_nones=False):
    # Create a directed graph using Graphviz
    dot = Digraph(comment='The Scene Graph')
    dot.attr(rankdir='TB') # Set the graph layout to flow from top to bottom

    # Graph label == scan ID
    dot.attr(label=scan_id)
    dot.attr('node', shape='oval', fontname='Sans')  # Default node attributes (oval shape, Sans font)

    # Split the scan ID into two parts
    a = scan_id
    a, b = '-'.join(a.split('-')[:-1]), a.split('-')[-1]
    # g_colors = node_color_list  # {o['id']: o['ply_color'] for o in self.scene_graphs_val[a+'_'+b]['objects']}
    
    # Loop through the list of ground truth objects to add nodes to the graph
    for index in range(len(objects_gt)):
        id = str(index) # Use the index as the node ID
        dot.attr('node', fillcolor=objects_color[index], style='filled')  # Set node color and fill style

        # Get the object name
        pred = objects_gt[index]
        pred = pred if pred != 'socket' else 'wall'
        # Add a node to the graph with the specified ID and label (object name)
        dot.node(id, pred)

    # Set edge attributes (black color, filled style, Sans font)
    dot.attr('edge', fontname='Sans', color='black', style='filled')
    # Loop through the ground truth predicates and edges to add relationships to the graph
    for i, (gt_predicate, edge) in enumerate(zip(gt_predicates, edges)):
        # Skip processing if the ground truth predicate is 'none'
        if gt_predicate == "none":
            break

        # Extract source and destination nodes from the edge
        # Get the predicted relationship from the model
        s, o = edge[:2]
        p_s1 = predicates1[i]
        p_s2 = predicates2[i]

        # Check if the predicted relationship is irrelevant and skip if include_nones is False
        if np.array([none_p in p_s1 for none_p in ['and',  'unrelated', 'not', 'none']]).any() and not include_nones:
            if p_s1 in ['', 'and', ' ', 'unrelated', 'not', 'none']:
                continue
        if np.array([none_p in p_s2 for none_p in ['and',  'unrelated', 'not', 'none']]).any() and not include_nones:
            if p_s2 in ['', 'and', ' ', 'unrelated', 'not', 'none']:
                continue
        # Create a label showing both the ground truth and model-predicted relationships
        label1 = model1 + ": " + p_s1 + "\n" + model2 + ": " + p_s2
        
        # Add an edge between the source and destination nodes with the label
        dot.edge(str(s.item()), str(o.item()), label1) #, color='blue')

        
    # Render the graph to a PNG file and clean up temporary files
    dot.render(filename, format="png", cleanup=True)

##### 4Bit vs 32Bit

In [14]:
# Load  objects.json file and get the list of scans
with open('/Volumes/projects/open3dsg/data/3RScan/3DSSG/objects.json') as file:
    objects_json = json.load(file)['scans']

model1 = 'blip' 
result_type1 = '4bit'

model2 = 'blip'
result_type2 = '32float'
scan_id = 'c92fb57e-f771-2064-86e4-5f4c7c77a8c7'
graph_filename = f"graphs_presentation/{scan_id}_{model1}" # Define the graph image name

# ----------------  FIRST GRAPH ----------------
data_dict = dataset[scan_id2idx.get(scan_id)]
for obj in objects_json:
    if obj['scan'] == scan_id:
        object_dict = obj
# Load results.json file
results_json1 = json.load(open(os.path.join(f"../results/{result_type1}/results_{scan_id}_{model1}.json")))
generated_texts1 = [result['output'] for result in results_json1]
objects_gt1, gt_predicates1, predicates1, objects_color1 = process_relationships(data_dict, model1, generated_texts1, results_json1, object_dict, eval_method='cos_similarity', visualitzation=False)

# ----------------  SECOND GRAPH ----------------
data_dict = dataset[scan_id2idx.get(scan_id)]
for obj in objects_json:
    if obj['scan'] == scan_id:
        object_dict = obj

# Load results.json file
results_json2 = json.load(open(os.path.join(f"../results/{result_type2}/results_{scan_id}_{model2}.json")))
generated_texts2 = [result['output'] for result in results_json2]
objects_gt2, gt_predicates2, predicates2, objects_color2 = process_relationships(data_dict, model2, generated_texts2, results_json2, object_dict, eval_method='cos_similarity', visualitzation=False)

# objects_gt1==objects_gt2 // gt_predicates1==gt_predicates2 // objects_color1==objects_color2
vis_graph_two_graphs_clip(scan_id=data_dict['scan_id'], 
                          objects_gt=objects_gt1, 
                          gt_predicates=gt_predicates1, 
                          objects_color=objects_color1, 
                          edges=data_dict['edges'][0][:len(predicates1)], 
                          model1=str(model1 + " " + result_type1), 
                          predicates1=predicates1, 
                          model2=str(model2 + " " + result_type2), 
                          predicates2=predicates2, 
                          filename=graph_filename, 
                          include_nones=True)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


##### LlaVA vs BLIP

In [16]:
# Load  objects.json file and get the list of scans
with open('/Volumes/projects/open3dsg/data/3RScan/3DSSG/objects.json') as file:
    objects_json = json.load(file)['scans']

model1 = 'blip' 
result_type1 = '4bit'

model2 = 'llava'
result_type2 = '4bit'
scan_id = '0cf75f4b-564d-23f5-899f-ff97701d3f3b'

# ----------------  FIRST GRAPH ----------------
data_dict = dataset[scan_id2idx.get(scan_id)]
for obj in objects_json:
    if obj['scan'] == scan_id:
        object_dict = obj
# Load results.json file
results_json1 = json.load(open(os.path.join(f"../results/{result_type1}/results_{scan_id}_{model1}.json")))
generated_texts1 = [result['output'] for result in results_json1]
objects_gt1, gt_predicates1, predicates1, objects_color1 = process_relationships(data_dict, model1, generated_texts1, results_json1, object_dict, eval_method='cos_similarity', visualitzation=False)

# ----------------  SECOND GRAPH ----------------
data_dict = dataset[scan_id2idx.get(scan_id)]
for obj in objects_json:
    if obj['scan'] == scan_id:
        object_dict = obj

# Load results.json file
results_json2 = json.load(open(os.path.join(f"../results/{result_type2}/results_{scan_id}_{model2}.json")))
generated_texts2 = [result['output'] for result in results_json2]
objects_gt2, gt_predicates2, predicates2, objects_color2 = process_relationships(data_dict, model2, generated_texts2, results_json2, object_dict, eval_method='cos_similarity', visualitzation=False)

# objects_gt1==objects_gt2 // gt_predicates1==gt_predicates2 // objects_color1==objects_color2
label1 = model1
label2 = model2
graph_filename = f"graphs_presentation/{scan_id}_{model1}_{model2}" # Define the graph image name
vis_graph_two_graphs_clip(scan_id=data_dict['scan_id'], 
                          objects_gt=objects_gt1, 
                          gt_predicates=gt_predicates1, 
                          objects_color=objects_color1, 
                          edges=data_dict['edges'][0][:len(predicates1)], 
                          model1=label1, 
                          predicates1=predicates1, 
                          model2=label2, 
                          predicates2=predicates2, 
                          filename=graph_filename, 
                          include_nones=True)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


### Scene Graph Selection

#### For 4bit vs 32float

In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image

def save_image_comparison(folder1, folder2, output_folder):
    """
    Saves pairs of images with the same name from two folders in a comparison image.

    Args:
        folder1 (str): Path to the first folder (e.g., '4bit').
        folder2 (str): Path to the second folder (e.g., '32float').
        output_folder (str): Path to the output folder where comparison images will be saved.
    """

    # Create the output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)

    # Get all image file names from both folders
    images_folder1 = {f for f in os.listdir(folder1) if f.endswith('.png')}
    images_folder2 = {f for f in os.listdir(folder2) if f.endswith('.png')}

    # Find common image names in both folders
    common_images = images_folder1.intersection(images_folder2)

    if not common_images:
        print("No common images found between the two folders.")
        return

    # Save each pair of images as a single comparison image
    for image_name in sorted(common_images):
        image1_path = os.path.join(folder1, image_name)
        image2_path = os.path.join(folder2, image_name)

        # Open the images
        image1 = Image.open(image1_path)
        image2 = Image.open(image2_path)

        # Plot the images side by side
        plt.figure(figsize=(12, 6))
        plt.subplot(1, 2, 1)
        plt.imshow(image1)
        plt.title("4bit")
        plt.axis('off')

        plt.subplot(1, 2, 2)
        plt.imshow(image2)
        plt.title("32float")
        plt.axis('off')

        # Add a title and save the figure
        plt.suptitle(f"Comparison of scan {image_name.split('_')[0]}")
        output_path = os.path.join(output_folder, f"comparison_{image_name}")
        plt.savefig(output_path, bbox_inches='tight')
        plt.close()

        print(f"Saved comparison image: {output_path}")

# Example usage:
folder_4bit = "/Volumes/scratch/alegretelena/LLaVA-3D/open3dsg/4bit_graph"
folder_32float = "/Volumes/scratch/alegretelena/LLaVA-3D/open3dsg/32float_graph"
output_folder = "/Volumes/scratch/alegretelena/LLaVA-3D/open3dsg/comparison_graphs"

save_image_comparison(folder_4bit, folder_32float, output_folder)

#### Prompt vs Original Prompt

In [ ]:
def save_image_comparison(folder1, folder2, output_folder):
    """
    Saves pairs of images with the same name from two folders in a comparison image.

    Args:
        folder1 (str): Path to the first folder (e.g., '4bit').
        folder2 (str): Path to the second folder (e.g., 'prompt_graphs').
        output_folder (str): Path to the output folder where comparison images will be saved.
    """

    # Create the output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)

    # Get all image file names from both folders
    images_folder1 = {f for f in os.listdir(folder1) if f.endswith('.png')}
    images_folder2 = {f for f in os.listdir(folder2) if f.endswith('.png')}

    # Find common image names in both folders
    common_images = images_folder1.intersection(images_folder2)

    if not common_images:
        print("No common images found between the two folders.")
        return

    # Save each pair of images as a single comparison image
    for image_name in sorted(common_images):
        image1_path = os.path.join(folder1, image_name)
        image2_path = os.path.join(folder2, image_name)

        # Open the images
        image1 = Image.open(image1_path)
        image2 = Image.open(image2_path)

        # Plot the images side by side
        plt.figure(figsize=(12, 6))
        plt.subplot(1, 2, 1)
        plt.imshow(image1)
        plt.title("Original Prompt 4bit")
        plt.axis('off')

        plt.subplot(1, 2, 2)
        plt.imshow(image2)
        plt.title("Prompt Engineering 4bit")
        plt.axis('off')

        # Add a title and save the figure
        plt.suptitle(f"Comparison of scan {image_name.split('_')[0]}")
        output_path = os.path.join(output_folder, f"comparison_{image_name}")
        plt.savefig(output_path, bbox_inches='tight')
        plt.close()

        print(f"Saved comparison image: {output_path}")

# Example usage:
folder_4bit = "/Volumes/scratch/alegretelena/LLaVA-3D/open3dsg/4bit_graph"
folder_prompt = "/Volumes/scratch/alegretelena/LLaVA-3D/open3dsg/prompt_graphs"
output_folder = "/Volumes/scratch/alegretelena/LLaVA-3D/open3dsg/comparison_graphs_prompt_vs_nonprompt"

save_image_comparison(folder_4bit, folder_prompt, output_folder)

#### Blip vs Llava

In [ ]:
def save_scanid_image_comparisons(folder, output_folder):
    """
    Finds images with the same scan ID but different models, plots them side by side, and saves the comparison.

    Args:
        folder (str): Path to the folder containing images (e.g., 'graphs').
        output_folder (str): Path to the output folder where comparison images will be saved.
    """

    # Create the output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)

    # Get all PNG images in the folder
    images = [f for f in os.listdir(folder) if f.endswith('.png')]

    # Group images by scan ID
    scan_groups = {}
    for image in images:
        # Extract the scan ID (everything before the first underscore)
        match = re.match(r"(.*?)_(.*?)\.png", image)
        if match:
            scan_id = match.group(1)
            if scan_id not in scan_groups:
                scan_groups[scan_id] = []
            scan_groups[scan_id].append(image)

    # Process each scan group with more than one image
    for scan_id, image_list in scan_groups.items():
        if len(image_list) > 1:
            # Open and plot the images side by side
            plt.figure(figsize=(12, 6))

            for i, image_name in enumerate(sorted(image_list)):
                image_path = os.path.join(folder, image_name)
                image = Image.open(image_path)

                plt.subplot(1, len(image_list), i + 1)
                plt.imshow(image)
                plt.title(image_name.split('_')[1].split('.')[0])  # Show the model name in the title
                plt.axis('off')

            # Save the comparison image
            output_path = os.path.join(output_folder, f"comparison_{scan_id}.png")
            plt.suptitle(f"Comparison of scan {scan_id}")
            plt.savefig(output_path, bbox_inches='tight')
            plt.close()

            print(f"Saved comparison image: {output_path}")

# Example usage:
folder_path = "/Volumes/scratch/alegretelena/LLaVA-3D/open3dsg/4bit_graph"
output_folder = "/Volumes/scratch/alegretelena/LLaVA-3D/open3dsg/comparison_graphs_llava_vs_blip"

save_scanid_image_comparisons(folder_path, output_folder)
